# สำรวจข้อมูล NHANES 2017-2018

เริ่มจากดูข้อมูล 3 ไฟล์ที่โหลดมา คือ DEMO (อายุ เพศ), BMX (ค่าวัดร่างกาย) และ DXX (ผลสแกน DXA)

## 1. โหลดข้อมูลดิบ

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RAW = "../data/raw"
df_demo = pd.read_sas(f"{RAW}/DEMO_J.XPT")
df_bmx = pd.read_sas(f"{RAW}/BMX_J.XPT")
df_dxa = pd.read_sas(f"{RAW}/DXX_J.xpt")

print("DEMO", df_demo.shape, "| BMX", df_bmx.shape, "| DXA", df_dxa.shape)

DEMO (9254, 46) | BMX (8704, 21) | DXA (5114, 93)


## 2. Demographics (DEMO_J)

In [2]:
key = ["SEQN", "RIDAGEYR", "RIAGENDR"]
print(df_demo[key].head(), "\n")
print("missing\n", df_demo[key].isna().sum(), "\n")
print(df_demo["RIDAGEYR"].describe(), "\n")
print(df_demo["RIAGENDR"].value_counts())

      SEQN  RIDAGEYR  RIAGENDR
0  93703.0       2.0       2.0
1  93704.0       2.0       1.0
2  93705.0      66.0       2.0
3  93706.0      18.0       1.0
4  93707.0      13.0       1.0 

missing
 SEQN        0
RIDAGEYR    0
RIAGENDR    0
dtype: int64 

count    9.254000e+03
mean     3.433423e+01
std      2.550028e+01
min      5.397605e-79
25%      1.100000e+01
50%      3.100000e+01
75%      5.800000e+01
max      8.000000e+01
Name: RIDAGEYR, dtype: float64 

RIAGENDR
2.0    4697
1.0    4557
Name: count, dtype: int64


อายุต่ำสุดออกมาเป็น 5.4e-79 ดูแปลกๆ เลยลองดูแถวที่อายุต่ำกว่า 1 ปี

In [3]:
infants = df_demo["RIDAGEYR"] < 1
print(df_demo.loc[infants, ["SEQN", "RIDAGEYR", "RIDAGEMN", "RIDEXAGM"]].head())
print(f"\nทารกอายุต่ำกว่า 1 ปี {infants.sum()} คน = {infants.mean() * 100:.2f}% ของ sample")

        SEQN      RIDAGEYR  RIDAGEMN  RIDEXAGM
7    93710.0  5.397605e-79      11.0      13.0
45   93748.0  5.397605e-79       7.0       8.0
83   93786.0  5.397605e-79       3.0       NaN
151  93854.0  5.397605e-79       8.0       8.0
162  93865.0  5.397605e-79       4.0       4.0

ทารกอายุต่ำกว่า 1 ปี 357 คน = 3.86% ของ sample


ไม่ได้เป็นข้อมูลผิด NHANES เก็บอายุของทารกเป็นเดือนไว้อีกคอลัมน์ ค่านี้เลยเป็นแค่เศษทศนิยม

จะใช้เฉพาะผู้ใหญ่ 18-59 ปี เพราะเด็กกับผู้สูงอายุมีร่างกายต่างจากวัยทำงานค่อนข้างมาก

## 3. Body Measures (BMX_J)

In [4]:
bm = ["BMXWT", "BMXHT", "BMXBMI"]
print("missing\n", df_bmx[bm].isna().sum(), "\n")
print(df_bmx[bm].describe())

missing
 BMXWT     124
BMXHT     688
BMXBMI    699
dtype: int64 

             BMXWT        BMXHT       BMXBMI
count  8580.000000  8016.000000  8005.000000
mean     65.138508   156.593401    26.577502
std      32.890754    22.257858     8.260724
min       3.200000    78.300000    12.300000
25%      43.100000   151.400000    20.400000
50%      67.750000   161.900000    25.800000
75%      85.600000   171.200000    31.300000
max     242.600000   197.700000    86.200000


## 4. รวม Demo กับ BMX และเอาเฉพาะ 18-59 ปี

Join ด้วย `SEQN` เอาเฉพาะคนที่มีทั้งอายุ เพศ และค่าวัดร่างกาย

In [5]:
df = df_demo.merge(df_bmx, on="SEQN", how="inner")
print("หลัง merge", df.shape, "| SEQN ซ้ำ", df["SEQN"].duplicated().sum())

df_adult = df[(df["RIDAGEYR"] >= 18) & (df["RIDAGEYR"] <= 59)].copy()
print("ผู้ใหญ่ 18-59 ปี", df_adult.shape, "\n")
print("missing\n", df_adult[bm].isna().sum(), "\n")
print(df_adult[bm + ["RIDAGEYR"]].describe())

หลัง merge (8704, 66) | SEQN ซ้ำ 0
ผู้ใหญ่ 18-59 ปี (3515, 66) 

missing
 BMXWT     40
BMXHT     40
BMXBMI    44
dtype: int64 

             BMXWT        BMXHT       BMXBMI     RIDAGEYR
count  3475.000000  3475.000000  3471.000000  3515.000000
mean     83.544691   167.274331    29.754278    38.325462
std      24.271417     9.982677     7.881479    12.538565
min      36.200000   138.300000    14.800000    18.000000
25%      66.300000   159.700000    24.200000    28.000000
50%      79.400000   166.700000    28.400000    38.000000
75%      96.400000   174.700000    33.700000    50.000000
max     242.600000   195.800000    86.200000    59.000000


## 5. DXA scan (DXX_J)

DXA แยกมวลไขมัน กล้ามเนื้อ และกระดูกได้ เลยใช้เป็นค่าจริงที่ model ต้องทายตาม

คำนวณ Fat % ทันทีไม่ได้ ต้องเปิด codebook ดูนิยามตัวแปรก่อน (`DXDTOFAT` กับ `DXDTOLE` เป็นหน่วยกรัม)

In [6]:
for k in ["FAT", "LE", "TOT"]:
    print(k, [c for c in df_dxa.columns if k in c])

print("\nmissing\n", df_dxa[["DXDTOFAT", "DXDTOLE"]].isna().sum())
print("มีไขมันและ lean ครบ", df_dxa[["DXDTOFAT", "DXDTOLE"]].notna().all(axis=1).sum(), "จาก", len(df_dxa))

FAT ['DXXHEFAT', 'DXXLAFAT', 'DXXLLFAT', 'DXXRAFAT', 'DXXRLFAT', 'DXXTRFAT', 'DXDSTFAT', 'DXDTOFAT']
LE ['DXDHELE', 'DXDLALE', 'DXDLLLE', 'DXDRALE', 'DXDRLLE', 'DXDTRLE', 'DXDSTLE', 'DXDTOLE']
TOT ['DXDHETOT', 'DXDLATOT', 'DXDLLTOT', 'DXDRATOT', 'DXDRLTOT', 'DXDTRTOT', 'DXDSTTOT', 'DXDTOTOT']

missing
 DXDTOFAT    1456
DXDTOLE     1294
dtype: int64
มีไขมันและ lean ครบ 3658 จาก 5114
